## 1. Library Imports

In [95]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from pandas.plotting import scatter_matrix

## 2. Data Loading

In [96]:
# training = pd.read_parquet("data/train_laps.parquet", filters=[("race_id", "==", "2023_01")]) 

training = pd.read_parquet("train_laps.parquet")
test = pd.read_parquet("test_laps.parquet")  

## 3. EDA

In [97]:
# Important info (data description from kaggle): 

# Main Dataset
# The main dataset is in training_laps.parquet and test_laps.parquet. 
# They were generated from data on Formula 1 races 2018-2026: 95 training races, 92 test races. 
# The test races are incomplete : Each test race is cut at a cutoff lap. 
# The cutoff lap was selected when the leader finished a lap between 30 % and 50 % of the race (uniformly at random). 
# It used the column "race_time_s"; you can use this too to generate cuts. 
# Every car shows only the laps it had finished by then, plus the oracle columns. 
# You will be scored on the missing part.

# Oracle columns
# Some future events may influence tyre strategy. 
# Teams often make strategies for different scenarios. 
# For you to be able to use at least a little bit of this information, 
# we have added some "look-ahead" columns that let you know the future: safety car in n+14 laps, for example. 
# Or rain. 
# These can be useful for wet races, amongst other things.

### 3.1 Quick Look (First look to get an idea)

In [98]:
# Info on the training data
training.info() 
# the 'next_compound' column here (column n°17) is the one we want to predict. It's not present in the test data

<class 'pandas.DataFrame'>
RangeIndex: 104757 entries, 0 to 104756
Data columns (total 72 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   lap_id                           104757 non-null  str    
 1   race_id                          104757 non-null  str    
 2   year                             104757 non-null  int64  
 3   round                            104757 non-null  int64  
 4   race_name                        104757 non-null  str    
 5   circuit                          104757 non-null  str    
 6   circuit_id                       104757 non-null  str    
 7   date                             104757 non-null  object 
 8   driver_code                      104757 non-null  str    
 9   driver_number                    104757 non-null  Int64  
 10  team                             104754 non-null  str    
 11  circuit_length_km                104757 non-null  float64
 12  turns        

In [99]:
training.isna().sum()

## Comments:
## - There's a lot of missing data. 
## - Ask TA if we can just use outside data to fill it 
##  (since this is considered cheating : "Using data beyond that provided in this competition. 
##  Notably, looking up the data of races in the test sets to improve your score in any way")
##  Update: he said no about outside data, but we're allowed to use the given data to fill in the missing values 
## - The goal is to fill as much missing values as possible using the data we have (this applies for both traning and test data)
## - The column 'next_compound' (the one we want to predict) has 3566 missing values (!!!!!!!!)

lap_id                       0
race_id                      0
year                         0
round                        0
race_name                    0
                         ...  
virtual_sc_Lplus14       22536
red_flag_Lplus14         22536
rainfall_Lplus14         22536
wind_speed_Lplus14_ms    22536
pressure_Lplus14_mbar    22536
Length: 72, dtype: int64

I tried to do some plotting to have an idea about the correlation between the data (I tried running the scatter_matrix function and training.corr(), but I got errors).
- I got an error when running training.corr() because of the column format (the column date has is a datetime.date object column, this was mainly the issue behind the error)
- For scatter_matrix(), the issue is the boolean columns.

I used claude to compute the correlation between variables whatever their type is, and especially the correlation with the 'next_compound' column (we have to bear in mind that the missing values haven't been dealt with yet)

In [100]:
def cramers_v(x, y):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    if d["x"].nunique() < 2 or d["y"].nunique() < 2:
        return np.nan
    ct = pd.crosstab(d["x"], d["y"])
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(kc - 1, rc - 1)) if min(kc - 1, rc - 1) > 0 else np.nan

def corr_ratio(cat, num):
    d = pd.DataFrame({"c": cat, "n": num}).dropna()
    if d["c"].nunique() < 2:
        return np.nan
    g = d.groupby("c")["n"].agg(["count", "mean"])
    ss_between = (g["count"] * (g["mean"] - d["n"].mean()) ** 2).sum()
    ss_total = ((d["n"] - d["n"].mean()) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan

def association_matrix(data, cat_cols, num_cols):
    cols = list(num_cols) + list(cat_cols)
    out = pd.DataFrame(np.nan, index=cols, columns=cols)
    for i, a in enumerate(cols):
        for b in cols[i:]:
            a_cat, b_cat = a in cat_cols, b in cat_cols
            if a == b:          v = 1.0
            elif a_cat and b_cat: v = cramers_v(data[a], data[b])
            elif a_cat:         v = corr_ratio(data[a], data[b])
            elif b_cat:         v = corr_ratio(data[b], data[a])
            else:               v = data[a].corr(data[b])
            out.loc[a, b] = out.loc[b, a] = v
    return out

# columns to leave out (identifiers / duplicates)
drop = ["lap_id", "date", "race_id", "circuit_id", "result_status"]

cat_cols = [c for c in training.select_dtypes(exclude=["number", "boolean", "bool"]).columns]
num_cols = list(training.select_dtypes(include=["number", "boolean", "bool"]).columns)

data = training.copy()
for c in num_cols: data[c] = data[c].astype("float64")   # fixes the pd.NA errors
for c in cat_cols: data[c] = data[c].astype("object")

data = data.sample(15000, random_state=0)                # speed: the full set is slow
am = association_matrix(data, cat_cols, num_cols)

# 1) What matters most for the target (according to Claude)
print(am["next_compound"].drop("next_compound").abs().sort_values(ascending=False).head(20)) 
## We only get 20 correlations. I ran the same command without .head(20) to get all the correlations in the next cell

'''
# 2) Full heatmap
plt.figure(figsize=(22, 18))
sns.heatmap(am, cmap="viridis", vmin=0, vmax=1)
plt.show()
'''

laps_remaining        0.727841
lap_number            0.721399
race_time_s           0.719492
stint                 0.557815
date                  0.423342
race_id               0.423342
compound              0.341231
race_name             0.302117
circuit_id            0.299643
circuit               0.299643
rainfall_Lplus14      0.267800
laps_down             0.261454
humidity_pct          0.240177
gap_to_leader_s       0.234776
tyre_life_laps        0.230605
track_temp_c          0.217161
air_temp_c            0.198830
pit_lane_time_s       0.191925
rainfall              0.188005
safety_car_Lplus14    0.178549
Name: next_compound, dtype: float64


'\n# 2) Full heatmap\nplt.figure(figsize=(22, 18))\nsns.heatmap(am, cmap="viridis", vmin=0, vmax=1)\nplt.show()\n'

In [101]:
# Correlation between the 'next_compound' column and the other columns
pre_missingvalues_filling = am["next_compound"].drop("next_compound").abs().sort_values(ascending=False)
pre_missingvalues_filling
## WE STILL HAVEN'T TOUCHED THE MISSING VALUES
## So the values might sometimes not make sense

laps_remaining           0.727841
lap_number               0.721399
race_time_s              0.719492
stint                    0.557815
date                     0.423342
                           ...   
drive_through_pending    0.018719
black_orange_flag        0.018616
lap_time_s               0.018354
stop_go_pending          0.014388
lap_id                        NaN
Name: next_compound, Length: 71, dtype: float64

In [102]:
# Info on the test data
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 40901 entries, 0 to 40900
Data columns (total 71 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   lap_id                           40901 non-null  str    
 1   race_id                          40901 non-null  str    
 2   year                             40901 non-null  int64  
 3   round                            40901 non-null  int64  
 4   race_name                        40901 non-null  str    
 5   circuit                          40901 non-null  str    
 6   circuit_id                       40901 non-null  str    
 7   date                             40901 non-null  object 
 8   driver_code                      40901 non-null  str    
 9   driver_number                    40901 non-null  Int64  
 10  team                             40900 non-null  str    
 11  circuit_length_km                40901 non-null  float64
 12  turns                        

In [103]:
test.isna().sum()
# We get the same result if we run the next 2 lines
#test.isnull().any() # returns a boolean if the column contains null variables or not (True = contains & False = does not contain)
#test.isnull().sum()

## Comments:
## - Here there's only 1 missing value in the column "team". 
## - And 1 missing value in the 'driving_time_s'
## 

lap_id                     0
race_id                    0
year                       0
round                      0
race_name                  0
                        ... 
virtual_sc_Lplus14       181
red_flag_Lplus14         181
rainfall_Lplus14         181
wind_speed_Lplus14_ms    181
pressure_Lplus14_mbar    181
Length: 71, dtype: int64

### 3.2 Dealing with missing data

#### 3.2.1.1 Training Dataset (version 1)

##### Missing teams

COLUMN : team

DESCRIPTION : Team name as used that season.

In [104]:
# There's 3 missing values in the column "team". 
# By figuring out who's the driver and which year was it, we can fill out those values (no outside data needed, just some data manipulation)

training[training['team'].isna()] # 3 missing values

## - Alonso (2018, Race 18, 5th lap)
## - Grosjean (2018, Race 18, 12th lap)
## - Ricciardo (2018, Race 18, 49th lap)


# The goal now is to figure out which team each one of those drivers belonged to during that race, 
# and use this to fill the missing value

training[(training['race_id'] == '2018_18') & (training['driver_code'].isin(['ALO','GRO', 'RIC']))]

## Alonso was at McLaren,
## GrosJean was at Haas F1 Team,
## and Ricciardo was at Red Bull Racing


# Fill the missing values with the actual teams

replacements = pd.Series(['McLaren', 'Haas F1 Team', 'Red Bull Racing'], index=[8147, 8316, 8748]) 
## Those are the row indexes to make sure there's no mix up when it comes to puting what team where
training['team'] = training['team'].fillna(replacements)

# Just to check that the values where indeed filled 
training[training['team'].isna()] # If there's nothing displayed, then they're filled

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar


This was th easiest to fill, since we don't have to take in consideration 1000 things at the same time. 

All that we had to do is figure out, during the specific race in which the value was missing, what was the driver's team.

##### Missing future compound

COLUMN : next_compound

DESCRIPTION : Target (train only), as if this lap L were the car's last visible lap: the tyre the car drives after its next pit stop after lap  L (a stop on lap L itself is already visible and does not count). SOFT / MEDIUM / HARD = softest / middle / hardest of the three dry compounds brought to that race; WET_OR_INTER = intermediate or full wet tyre; NONE = no further stop, the car drives to the flag. NA if the car retires without another stop, or the tyre after the stop is unknown.

In [105]:
training[training['next_compound'].isna()] # There's 3566 missing values here

# I have a theory that theyre missing because the drivers in question didn't finish the race 
# (race not finished = no pit stop)

# Checking the theory
training.loc[training['next_compound'].isna(), 'result_status'].value_counts()

# Theory confirmed

# And since the column's description explicitly mentions that the values are NA 
# if the car retires without another stop, or the tyre after the stop is unknown; 
# that's how we'll deal with the missing values. 
# We'll fill them with NA

training['next_compound'] = training['next_compound'].fillna('NA')
training[training['next_compound'].isna()] # Checking that there's no missing values left

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar


##### Missing lap time

COLUMN : lap_time_s

DESCRIPTION : Time from crossing the timing line to crossing it again.

In [106]:
training[training['lap_time_s'].isna()] # There's 205 missing values here
# Same theory: unfinished race
# Subplot: The lap in question (where the lap time is missing) is the lap where the drivers faced the issue
# that made them stop the race (retire from the race due to whatever reason)

training.loc[training['lap_time_s'].isna(), 'result_status'].value_counts() 
# 202 who have a missing value here didn't finish the race for x reason
# The 3 others finished 1 lap behind

training[(training['lap_time_s'].isna()) & (training['result_status'] == '+1 Lap')]

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
20128,2019_17_PER_052,2019_17,2019,17,Japanese Grand Prix,Suzuka Circuit,suzuka,2019-10-13,PER,11,...,False,4.1,295.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
62566,2023_14_HUL_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,HUL,27,...,False,1.7,161.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
62719,2023_14_MAG_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,MAG,20,...,False,1.7,161.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


We wanted to know if those rows are the last registerd rows for the concerned drivers in the concerned races. We used Claude to filter that and got this:

Almost all of them are: 186 of the 205 missing lap_time_s rows are the driver’s last recorded lap in that race. The 205 rows belong to 187 driver-race combinations.


| Case              | Driver-races | Missing rows |
| :---------------- | :------: | ----: |
| Only the last lap is missing        |   168   | 168 |
| The last lap and the one before it are missing         |   18   | 36 |
| The missing lap is not near the end    |  1   | 1 |


The 19 rows that are not the last lap are made up of two groups:

18 rows are the second-to-last lap. They all come from the 2026 season (for example 2026_01_BOT_016 with last lap 17, or 2026_15_STR_008 with last lap 9). Each driver’s final lap is missing too. These are all Retired, so the driver had no timed lap just before the retirement.
1 row is a real exception: 2018_18_RIC_009. Ricciardo’s lap 9 is missing, but his race lasted 49 laps (status Battery), and his last lap has a time.

What the missing rows are

The last-lap pattern covers 186 rows. Among all 1,916 driver-races, 186 final laps have no time. That is only about 10%, so most drivers’ last lap does have a time. The statuses of the 205 rows are mostly retirements: Retired (92), Collision (42), Accident (14), Engine (11), Did not start (6), and so on. We saw that some +1 Lap finishers also have an untimed last lap, which is the PER and HUL/MAG case.

So your theory holds for 90% of the rows. You can update the comment to say “missing lap time = the last recorded lap (or the one before it) of a driver who retired”. 2018_18_RIC_009 is the one row that needs separate handling. To list the exceptions yourself:

In [107]:
# What I am going to do is fill the 205 missing values with NA

training['lap_time_s'] = training['lap_time_s'].fillna('NA') 
# Not possible. 
# The column has a dtype float64. 
# I cant fill the missing values with a string (text)
# Either we leave the missing values empty or we transform the column into an object

C:\Users\laila\AppData\Local\Temp\ipykernel_16676\1414240764.py:3: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  training['lap_time_s'] = training['lap_time_s'].fillna('NA')


##### Missing driving time 

COLUMN : driving_time_s

DESCRIPTION : Time the car was actually driving on this lap.

In [108]:
training[training['driving_time_s'].isna()] # 3 missing values

# The 3 cases where there isn't a driving_time_s is because the drivers in question haven't finished the race: 
# Alonso had a collision in lap 5 
# GrosJean had a collision damage in lap 12
# Ricciardo had battery issues in lap 49

# There's also missing values in the 'next_compound column (the one we want to predict), which makes some sense since they DNF'd, 
# so there's no pit stop.

# DECISION:
# I won't fill them because
# - I think it might be complicated to compute it since idk at which point of the lap they had their issues.
# - It might be something the model can learn/we can teach it (that sometimes, missing values are caused by unfinished races)


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
8147,2018_18_ALO_005,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ALO,14,...,False,1.9,89.0,100.133,False,False,False,False,1.0,1005.8
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,3.3,121.0,99.500,False,False,False,False,1.4,1005.8
8748,2018_18_RIC_049,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,False,2.2,107.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


##### Missing sector 1

COLUMN : sector1_s

DESCRIPTION : Time for the first of three track sections.

In [109]:
training[training['sector1_s'].isna()] # 2322 missing values

# I think I'll leave this one untouched, because since this is a column related to timing, 
# I don't want to fuck it up. 
# Add to that that we don't know what's the sector en question, so not to bullshit anything

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
58,2018_01_BOT_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,BOT,77,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
116,2018_01_ERI_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
122,2018_01_GAS_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
136,2018_01_GRO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GRO,8,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104646,2026_15_SAI_001,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,SAI,55,...,False,0.6,63.0,107.810,False,False,False,False,0.3,1023.1
104697,2026_15_STR_001,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.6,63.0,107.810,False,False,False,False,0.3,1023.1
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0
104705,2026_15_STR_009,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,2.7,22.0,107.109,False,False,False,False,1.3,1022.9


##### Missing sector 2

COLUMN : sector2_s

DESCRIPTION : Time for the second section.

In [110]:
# Same thing applies here

##### Missing sector 3

COLUMN : sector3_s

DESCRIPTION : Time for the third section.

In [111]:
# And here

##### Missing speed i1

COLUMN : speed_i1_kmh

DESCRIPTION: Speed at intermediate measuring point 1.

In [112]:
# And here

##### Missing speed i2

COLUMN : speed_i2_kmh

DESCRIPTION : Speed at intermediate measuring point 2

In [113]:
# And here

##### Missing speed finish line

COLUMN : speed_finish_line_kmh

DESCRIPTION : Speed at the finish line

In [114]:
# And here

##### Missing speed trap

COLUMN : speed_trap_kmh

DESCRIPTION : Speed at the speed trap (end of the longest straight)

In [115]:
# And here

##### Missing positions

COLUMN: position

DESCRIPTION : Running position at the end of the lap, 1 = leading.

In [116]:
training[training['position'].isna()] # 175 missing values

# I couldn't get a lot of details, 
# so what I did is that I looked at the reasons why the drivers didn't finish the race 
# (I looked at the result status)
# The code in the next cell allowed me to do that

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,2.2,80.0,106.492,False,False,False,False,2.8,1022.0
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [117]:
training.loc[training['position'].isna(), 'result_status'].value_counts()
# This is the result_status for those that have missing values in the 'position' column.

# Turns out that 3 of them have 'result_status' == +1 Lap, so I'll see if I can just figure out their position
# The 172 others had issues (the coresponding 'result_status' column had values like Engine, Retired...), 
# so it makes sense that there's no values in the 'position' column (race not finished = no position). 
# We can deal with it by putting DNF, DNS, ... as values in the 'position' column.

result_status
Retired             71
Collision           42
Accident            14
Engine              10
Power Unit           5
Did not start        4
Brakes               3
Hydraulics           3
Suspension           3
Spun off             3
+1 Lap               3
Puncture             2
Exhaust              2
Gearbox              2
Battery              2
Oil leak             1
Fuel pressure        1
Power loss           1
Collision damage     1
Transmission         1
Driveshaft           1
Name: count, dtype: int64

In [118]:
training[(training['position'].isna()) & (training['result_status'] == '+1 Lap')]

# So the 3 drivers with no position and a result status of +1 Lap are:
# Sergio Perez during the Japanese GP in 2019
# Nico Hulkenberg and Kevin Magnussen during the Italian GP in 2023


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
20128,2019_17_PER_052,2019_17,2019,17,Japanese Grand Prix,Suzuka Circuit,suzuka,2019-10-13,PER,11,...,False,4.1,295.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
62566,2023_14_HUL_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,HUL,27,...,False,1.7,161.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
62719,2023_14_MAG_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,MAG,20,...,False,1.7,161.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [119]:
# Decision
# I'll leave this column as is to not mess up.
# We could try to teach the model to look up for certain stuff before predicting
# If anyone has an idea to how to deal with this column be my guest


In [120]:
# Check this duplicate issue for all positions
for i in range(1, 23):
    dups = training[training["position"] == i].groupby(["race_id", "lap_number", 'position']).size()
    print(f"Position : {i}")
    print(dups[dups > 1])

Position : 1
race_id  lap_number  position
2020_09  8           1           2
2021_05  30          1           2
2024_12  1           1           2
dtype: int64
Position : 2
race_id  lap_number  position
2023_15  1           2           2
2023_20  1           2           2
2024_04  1           2           2
2024_08  1           2           2
dtype: int64
Position : 3
race_id  lap_number  position
2023_20  1           3           2
2024_04  1           3           2
2024_08  1           3           2
dtype: int64
Position : 4
race_id  lap_number  position
2023_20  1           4           2
2024_08  1           4           2
dtype: int64
Position : 5
race_id  lap_number  position
2024_08  1           5           2
dtype: int64
Position : 6
race_id  lap_number  position
2026_06  44          6           2
dtype: int64
Position : 7
Series([], dtype: int64)
Position : 8
race_id  lap_number  position
2023_08  54          8           2
dtype: int64
Position : 9
Series([], dtype: int64)
Positio

There's a few positions that have duplicates

##### Missing race time

COLUMN : race_time_s

DESCRIPTION : Race time so far: sum of this driver's lap times up to this lap

*Main Dataset*

*The main dataset is in training_laps.parquet and test_laps.parquet. They were generated from data on Formula 1 races 2018-2026: 95 training races, 92 test races. The test races are incomplete : Each test race is cut at a cutoff lap. The cutoff lap was selected when the leader finished a lap between 30 % and 50 % of the race (uniformly at random). It used the column "race_time_s"; you can use this too to generate cuts. Every car shows only the laps it had finished by then, plus the oracle columns. You will be scored on the missing part.*

In [121]:
training[training['race_time_s'].isna()] # 206 missing values

# I realised that for the columns 'driving_time_s' and 'lap_time_s', they had common values multiple times 
# (I only realised this now because since 'race_time_s' is based on the sum of the lap times, I was giving a quick look at the 'lap_time_s' column)
# So I want to check if for all the rows where there's values in both columns ('driving_time_s' and 'lap_time_s'), if the values are the same.
# That's what I'm doing in the next cell

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [122]:
'''# Check if for all the rows where there's values in both columns ('driving_time_s' and 'lap_time_s'), if the values are the same.
both = training[training["lap_time_s"].notna() & training["driving_time_s"].notna()].copy()

both["diff"] = both["driving_time_s"] - both["lap_time_s"]
both["same"] = np.isclose(both["driving_time_s"], both["lap_time_s"], atol=1e-3)

print(len(both))
print(both["same"].value_counts())

# Turns out that for 3637 case, they're not similar'''

'# Check if for all the rows where there\'s values in both columns (\'driving_time_s\' and \'lap_time_s\'), if the values are the same.\nboth = training[training["lap_time_s"].notna() & training["driving_time_s"].notna()].copy()\n\nboth["diff"] = both["driving_time_s"] - both["lap_time_s"]\nboth["same"] = np.isclose(both["driving_time_s"], both["lap_time_s"], atol=1e-3)\n\nprint(len(both))\nprint(both["same"].value_counts())\n\n# Turns out that for 3637 case, they\'re not similar'

In [123]:
'''both.loc[~both["same"], ["lap_id", "lap_number", "lap_time_s", "driving_time_s", "diff"]].head(20)'''

'both.loc[~both["same"], ["lap_id", "lap_number", "lap_time_s", "driving_time_s", "diff"]].head(20)'

In [124]:
# It's not going to be possible to compute the 'race_time_s', because - let's say we're on lap 2 - we'll need the 'lap_time_s' of lap 1 AND lap 2 
# (I manually did it using complete rows, that's how I know (Tutorial few lignes below)).
# So if we don't have the 'lap_time_s' of lap 2, we can't compute the 'race_time_s'.

# And we actually don't have the 'lap_time_s' for our missing values here 
# (except 1 row. 'lap_time_s' has 205 missing values and 'race_time_s' has 206 missing values)
# This is the only row where we might compute the 'race_time_s'.
# Other than that we'll have to stick with the missing values

# TUTO on how I did it:
# I used the following rows: 2018_01_ALO_001 & 2018_01_ALO_002
# lap_time_s for 2018_01_ALO_001 : 101.528 s
# lap_time_s for 2018_01_ALO_002 : 91.565 s
# race_time_s for 2018_01_ALO_001: 101.528 s (== lap_time_s for 2018_01_ALO_001 (which makes sense since it's only the first lap))
# race_time_s for 2018_01_ALO_002: 193.093 s  (== 101.528 + 91.565)

##### Missing gap to leader

COLUMN : gap_to_leader_s

DESCRIPTION : Seconds behind the leader at the end of this lap

In [125]:
training[training['gap_to_leader_s'].isna()]
# First thing we noticed is that those who had a missing gap to leader are driver who didn't finish the race/didn't even start.
# Theory is that the row which has a missing gap to leader is because it's the one where they were forced to retire

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [126]:
'''# Testing the theory on some drivers

## Case 1
training[(training['race_id'] == '2018_01') & (training['driver_code'].isin(['ERI','GAS', 'SIR']))] 
### Theory confirmed here. None of them completed the whole race, and during the last lap recorded, there's no gap to leader, 
### no time lap (which shows that they DNFd for some reason)

## Case 2
training[(training['race_id'] == '2018_04') & (training['driver_code'].isin(['BOT','GRO', 'HUL', 'OCO', 'RIC', 'SIR', 'VER']))] '''

"# Testing the theory on some drivers\n\n## Case 1\ntraining[(training['race_id'] == '2018_01') & (training['driver_code'].isin(['ERI','GAS', 'SIR']))] \n### Theory confirmed here. None of them completed the whole race, and during the last lap recorded, there's no gap to leader, \n### no time lap (which shows that they DNFd for some reason)\n\n## Case 2\ntraining[(training['race_id'] == '2018_04') & (training['driver_code'].isin(['BOT','GRO', 'HUL', 'OCO', 'RIC', 'SIR', 'VER']))] "

In [127]:
training.loc[training['gap_to_leader_s'].isna(), 'result_status'].value_counts() 
# According to this, none finished the race

result_status
Retired             92
Collision           42
Accident            14
Engine              11
Did not start        6
Power Unit           5
Brakes               4
Suspension           4
Gearbox              3
Exhaust              3
Hydraulics           3
Spun off             3
+1 Lap               3
Puncture             2
Battery              2
Steering             1
Oil leak             1
Fuel pressure        1
Power loss           1
Collision damage     1
Transmission         1
Wheel nut            1
Driveshaft           1
Front wing           1
Name: count, dtype: int64

In [128]:
training.loc[training['gap_to_leader_s'].isna(), 'position'].value_counts() 
# But according to this, they have a final position after the race, it's not missing

position
15    4
2     4
18    3
20    3
1     3
3     3
19    3
16    2
4     2
8     1
12    1
5     1
17    1
6     1
22    1
Name: count, dtype: Int64

In [129]:
training[(training['gap_to_leader_s'].isna()) & (training['position'] == 1)]
# issue: dans ces 3 cas, there's also another driver with position = 1

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
25127,2020_09_OCO_008,2020_09,2020,9,Tuscan Grand Prix,Autodromo Internazionale del Mugello,mugello,2020-09-13,OCO,31,...,False,2.0,347.0,83.331,False,False,False,False,3.2,993.6
33566,2021_05_BOT_030,2021_05,2021,5,Monaco Grand Prix,Circuit de Monaco,monaco,2021-05-23,BOT,77,...,False,0.3,173.0,74.993,False,False,False,False,0.5,1015.5
75824,2024_12_GAS_001,2024_12,2024,12,British Grand Prix,Silverstone Circuit,silverstone,2024-07-07,GAS,10,...,False,1.6,215.0,90.946,False,False,False,False,1.9,992.0


In [130]:
# We're looking for cases where more than 1 driver hve position == 1
# It's to see if the problem is only with the previous 3 drivers or not
dups = training[training["position"] == 1].groupby(["race_id", "lap_number"]).size()
dups[dups > 1]

race_id  lap_number
2020_09  8             2
2021_05  30            2
2024_12  1             2
dtype: int64

In [131]:
# We're looking for cases where more than 1 driver hve position == 1
# It's to see if the problem is only with the previous 3 drivers or not
dups = training[training["position"] == 1].groupby(["race_id", "lap_number", 'position']).size()
dups[dups > 1]

race_id  lap_number  position
2020_09  8           1           2
2021_05  30          1           2
2024_12  1           1           2
dtype: int64

##### Missing laps down

COLUMN : laps_down

DESCRIPTION : Laps behind the leader (0 = on the lead lap)

In [132]:
training[training['laps_down'].isna()] # 206 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


##### Missing gap to car ahead

Column: gap_to_car_ahead_s

Just realised something.

In the cases where drivers didn't finish the race, it doesn't make sense to fill the missing values with 0 BECAUSE it's also 0 when the car is P1 (they're first, there's no car ahead of them). So if both have a value of 0, I'm scared that the model might fuck up.

In [133]:
training[training['gap_to_car_ahead_s'].isna()] # 5964 missing values
# There's a few reasons why there's missing values in this column:
# - The driver's position is 1. Which means he's leading, which means there's no car ahead. 
#   In this case we could just replace the missing values with 0
# - The driver didn't finish the race fo a certain reason. Which means that at some point during the race, he stoped while the others are still racing. 
#   Which also means that there's also no car ahead of him (makes sense since he's out)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
160,2018_01_HAM_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
161,2018_01_HAM_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
162,2018_01_HAM_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104643,2026_15_RUS_049,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,False,0.6,353.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104644,2026_15_RUS_050,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,False,0.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104645,2026_15_RUS_051,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,False,0.9,305.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [134]:
# We'll start by filling the missing values of the drivers who's position == 1
mask = training['gap_to_car_ahead_s'].isna() & (training['position'] == 1)
training.loc[mask, 'gap_to_car_ahead_s'] = 0

In [135]:
# just to make sure that it was filled
x = training[training['position'] == 1]

## Realised something wrong: in 3 cases, the position is 1 but there IS a gap with the car ahead 
## (this gap is also = to the gap to leader)
## This means that either the actual position is 2, or the timing is wrong

In [136]:
training.loc[training['gap_to_car_ahead_s'].isna(), 'position'].value_counts()

position
2     5
15    4
18    3
20    3
3     3
19    3
16    2
4     2
8     1
12    1
5     1
17    1
6     1
22    1
Name: count, dtype: Int64

In [137]:
# Check the reasons why the drivers didn't finish
training.loc[training['gap_to_car_ahead_s'].isna(), 'result_status'].value_counts()

## There's 1 case were the result status is 'Finished', which means that the driver actually finished the race.
## What we'll do is that for the rest of the cases, we'll also fill the gap to car ahead with 0.
## For the result_status == Finished case, we'll see if we can figure out the gap. If not, we'll leave it as is (with a missing value).

result_status
Retired             92
Collision           43
Accident            14
Engine              11
Power Unit           5
Did not start        5
Suspension           4
Brakes               3
Gearbox              3
Exhaust              3
Hydraulics           3
Spun off             3
+1 Lap               3
Puncture             2
Collision damage     2
Battery              2
Steering             1
Oil leak             1
Fuel pressure        1
Power loss           1
Transmission         1
Driveshaft           1
Front wing           1
Finished             1
Name: count, dtype: int64

In [138]:
'''mask = training["gap_to_car_ahead_s"].isna()
mask &= ~(training["result_status"] == "Finished")   # skip the row that matches this
training.loc[mask, "gap_to_car_ahead_s"] = 0'''

'mask = training["gap_to_car_ahead_s"].isna()\nmask &= ~(training["result_status"] == "Finished")   # skip the row that matches this\ntraining.loc[mask, "gap_to_car_ahead_s"] = 0'

In [139]:
'''training.loc[training['gap_to_car_ahead_s'].isna(), 'result_status'].value_counts()

# We're almost done with filling the missing values of the 'gap_to_car_ahead_s' column. 
# Only thing left is the case where the driver actually finished the race'''

"training.loc[training['gap_to_car_ahead_s'].isna(), 'result_status'].value_counts()\n\n# We're almost done with filling the missing values of the 'gap_to_car_ahead_s' column. \n# Only thing left is the case where the driver actually finished the race"

In [140]:
# Getting more information on that case
training[training['gap_to_car_ahead_s'].isna()] 

## So now we know that it's Hamilton during the US GP (19th round) in 2022, during the 40th lap

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [141]:
training[(training['race_id'] == '2022_19') & (training['lap_number'] == 40)] 
# According to the data, during that race at lap 40, Hamilton was P2 and and Vettel was P1.
# The thing is, if we look at the 'gap_to_leader_s' column, we see that it's 0 for Hamilton and 0.22 for Vettel. 
# In the 'gap_to_car_ahead_s' column, it's also 0.22 for Vettel.
# In our opinion, this either means that there was a swap/typo in the actual position (Hamilton is actually P1 and Vettel is P2), 
# or when this data was collected, the positions weren't updated

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
51726,2022_19_ALB_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,ALB,23,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
51782,2022_19_ALO_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,ALO,14,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
51855,2022_19_GAS_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,GAS,10,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
51911,2022_19_HAM_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,HAM,44,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
51967,2022_19_LAT_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,LAT,6,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
52023,2022_19_LEC_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,LEC,16,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
52079,2022_19_MAG_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,MAG,20,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
52135,2022_19_MSC_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,MSC,47,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
52191,2022_19_NOR_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,NOR,4,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0
52247,2022_19_OCO_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,OCO,31,...,False,5.5,100.0,100.168,False,False,False,False,5.3,990.0


In [142]:
# What we'll do now is fill this last missing value
#training['gap_to_car_ahead_s'] = training['gap_to_car_ahead_s'].fillna(0)

##### Missing gap to car behind

Column: gap_to_car_behind_s

In [143]:
training[training['gap_to_car_behind_s'].isna()]  # 5964 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,3.5,295.0,88.245,False,False,False,False,3.5,997.0
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,2.2,278.0,121.319,True,True,False,False,2.2,996.9
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
219,2018_01_HAR_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
220,2018_01_HAR_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104701,2026_15_STR_005,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.580,False,False,False,False,0.0,1023.1
104702,2026_15_STR_006,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,1.1,191.0,107.572,False,False,False,False,0.5,1023.0
104703,2026_15_STR_007,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,1.1,254.0,107.465,False,False,False,False,2.2,1022.9
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


In [144]:
training.loc[training['gap_to_car_behind_s'].isna(), 'position'].value_counts()
# What i want to do is figure out what's the last position recorded for lap of each race
# In this case, we'll fill the missing values with 0 because if you're the last car, 
# there's no one behind you

position
20    1491
18    1307
19    1167
17     754
16     381
15     246
14     146
21      84
22      71
13      48
12      27
6       10
9       10
10       9
8        7
5        7
7        6
11       5
3        4
4        4
2        4
1        3
Name: count, dtype: Int64

In [145]:
# Just to check if in total, we have the 5964 missing values
training.loc[training['gap_to_car_behind_s'].isna(), 'position'].value_counts().sum()
# Nope, only 5791 have a recorded position at the end of the race
# which means that for the 173 other missing values, the driver didn't finish the race
# In this case, i might just leave the missing values as is

np.int64(5791)

##### Missing personal best

Column: personal_best

In [146]:
training[training['personal_best'].isna()] # 137 missing values
# keep in mind that it's a boolean column

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,False,1.0,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,False,1.2,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,2.2,80.0,106.492,False,False,False,False,2.8,1022.0
1542,2018_04_RIC_040,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,RIC,3,...,False,2.4,37.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,0.8,348.0,105.005,False,False,False,False,0.0,1023.1
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,2.1,249.0,106.520,False,False,False,False,0.9,1023.0
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,0.0,0.0,107.191,False,False,False,False,0.7,1023.0


##### Missing compound

Column: compound

In [147]:
training[training['compound'].isna()] # There's 27 missing values here

# For all of the 27 missing values, it's the 1st lap of the race 
# (not always the same race, but it's always the 1st lap)!
# What I want to do is have also the 2nd lap for these 27 cases.
# This will allow us to know what compound is each driver is on (unless they pit at that moment (doesn't make sense but maybe))
# Turns out they all have 'official_stop' = True
# I used Claude to do confirm that in the next cell

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1452,2018_04_RAI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,RAI,7,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
2051,2018_05_HAR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,HAR,28,...,True,1.4,96.0,81.778,False,False,False,True,2.4,1001.5
2504,2018_05_SIR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,SIR,35,...,True,1.4,96.0,81.778,False,False,False,True,2.4,1001.5
2920,2018_07_ERI_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,ERI,9,...,False,1.2,21.0,75.425,False,False,False,False,2.7,1017.9
3820,2018_07_VAN_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,VAN,2,...,False,1.2,21.0,75.425,False,False,False,False,2.7,1017.9


In [148]:
# the 27 cases (lap 1 rows with missing compound)
missing = training[training["compound"].isna()]

# build a (race_id, driver_code) key for them
keys = pd.MultiIndex.from_frame(missing[["race_id", "driver_code"]])

# keep laps 1 and 2 for those drivers in those races
mask = (
    pd.MultiIndex.from_frame(training[["race_id", "driver_code"]]).isin(keys)
    & training["lap_number"].isin([1, 2])
)

missing_laps_1_2 = (
    training[mask]
    .sort_values(["race_id", "driver_code", "lap_number"])
)

missing_laps_1_2

# Observations: 
# - Each 1st lap has official_stop = True
# - Each 2nd row has stint = 2, which confirms that for those 27 cases, they made a pit stop in the first lap
# And since we don't know what's the compound or what's its actual hardness, we leave it empty
# This applies for compound, coumpound_hardness and hardness scale

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
219,2018_01_HAR_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
941,2018_04_ALO_002,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.2,62.0,107.053,False,False,False,False,3.9,1021.9
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1041,2018_04_ERI_002,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.2,62.0,107.053,False,False,False,False,3.9,1021.9
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1237,2018_04_HAR_002,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.2,62.0,107.053,False,False,False,False,3.9,1021.9
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1350,2018_04_MAG_002,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.2,62.0,107.053,False,False,False,False,3.9,1021.9


##### Missing compound hardness

COLUMN : compound_hardness

DESCRIPTION : Actual dry tyre: C1 (hardest) to C6 (softest); 2018 uses that year's names (e.g. SUPERSOFT).

In [149]:
training[training['compound_hardness'].isna()]

# 'compound_hardness' has 4999 missing values.
# 27 out of the 4999 also have a missig value in the 'compound' column.
# For the rest (the other 4972 values), the 'compound' is either wet or intermediate.
# The hardness in this column only applies on the dry tyres
# And since the missing values in compound are either wet/inters or NA, 
# the compound hardness does not apply
# So as for now we'll leave the column as is

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86211,2025_13_VER_008,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,2.7,313.0,106.300,False,False,False,False,0.7,965.3
86212,2025_13_VER_009,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,2.0,275.0,106.433,False,False,False,False,1.2,965.3
86213,2025_13_VER_010,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,1.5,339.0,106.193,False,False,False,False,1.4,965.3
86214,2025_13_VER_011,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,1.9,355.0,106.199,False,False,False,False,1.6,965.3


##### Missing hardness scale

COLUMN : hardness_scale

DESCRIPTION : Which scale compound_hardness uses: 'C1-C6' or '2018 name'.

In [150]:
training[training['hardness_scale'].isna()] # 4999 missing values
#training.loc[training['hardness_scale'].isna(), 'compound'].value_counts()
# What we applied on 'compound_hardness' also applies here

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86211,2025_13_VER_008,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,2.7,313.0,106.300,False,False,False,False,0.7,965.3
86212,2025_13_VER_009,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,2.0,275.0,106.433,False,False,False,False,1.2,965.3
86213,2025_13_VER_010,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,1.5,339.0,106.193,False,False,False,False,1.4,965.3
86214,2025_13_VER_011,2025_13,2025,13,Belgian Grand Prix,Circuit de Spa-Francorchamps,spa-francorchamps,2025-07-27,VER,1,...,False,1.9,355.0,106.199,False,False,False,False,1.6,965.3


##### Missing tyre life

COLUMN : tyre_life_laps

DESCRIPTION : Laps driven on this set of tyres, including this lap.

In [151]:
training[training['tyre_life_laps'].isna()] # There's 27 missing values here

# At first glance, we can see that for those who the 'tyre_life_laps' is missing, 
# the tyre's compound is also missing 
# (compound, compound_hardness, hardness_scale are all missing).
# Another observation: for all of the 27 missing values, it's the 1st lap of the race 
# (not always the same race, but it's always the 1st lap)!
# This makes things very easy!! 
# Since it's the 1st lap of each of the races, it's also the tyre's 1st lap.
# So all we gotta do is fill the missing values with 1.
# Not true. They might be tires used during the Qualification. In this case, its life isn't 1.
# So as for now we'll leave the column as is

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
1452,2018_04_RAI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,RAI,7,...,False,3.0,92.0,106.722,False,False,False,False,2.0,1021.8
2051,2018_05_HAR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,HAR,28,...,True,1.4,96.0,81.778,False,False,False,True,2.4,1001.5
2504,2018_05_SIR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,SIR,35,...,True,1.4,96.0,81.778,False,False,False,True,2.4,1001.5
2920,2018_07_ERI_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,ERI,9,...,False,1.2,21.0,75.425,False,False,False,False,2.7,1017.9
3820,2018_07_VAN_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,VAN,2,...,False,1.2,21.0,75.425,False,False,False,False,2.7,1017.9


##### Missing pit lane time

COLUMN : pit_lane_time_s

DESCRIPTION : Time in the pit lane from entry to exit (about 20-30 s).

In [152]:
training[training['pit_lane_time_s'].isna()]

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104752,2026_15_VER_047,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,1.7,357.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104753,2026_15_VER_048,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,2.2,54.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104754,2026_15_VER_049,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,0.6,353.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104755,2026_15_VER_050,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,0.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [153]:
training[training['pit_lane_time_s'].isna()] # 101383 missing values 

# After a quick look, we realised that there's so much missing values because at that moment, 
# the driver didn't pit (no pit, no pit lane time, no value, we end up with missing values (tadaaaa))
# We realised this fact by looking at the official_stop column.
# And to confirm it, we ran the code in the next cell.

training.loc[training['pit_lane_time_s'].isna(), 'official_stop'].value_counts()

# We can see that for ALL the missing values, there was no pit stop. 
# At least we're sure that it's not someone who didn't fill the time for a pit stop procees

training.loc[training['pit_lane_time_s'].isna(), 'result_status'].value_counts()
# Turns out that filling those missing values isn't that simple.
# If the driver finished the race or has '+... Lap(s)' as result, we put 0.
# BUT if he didn't finish, things get more complicated : 
# If he didn't finish but could still drive the car to the garage, there must be a certain time 

result_status
Finished            60293
+1 Lap              27516
+2 Laps              5300
Retired              3306
Collision             660
Disqualified          541
+3 Laps               524
Collision damage      516
Engine                386
Accident              258
Power Unit            226
Brakes                201
Gearbox               199
Hydraulics            195
Suspension            157
Exhaust               105
Power loss            102
Puncture               88
Wheel                  70
Oil leak               68
Fuel pressure          61
Front wing             55
Turbo                  51
Lapped                 50
Rear wing              46
Undertray              46
Vibrations             45
Withdrew               45
Differential           40
+15 Laps               39
Mechanical             38
Spun off               37
Water leak             32
Wheel nut              30
Overheating            25
Battery                10
Transmission            9
Steering                

In [154]:
# Filling the missing values with 0 for those who finished
mask = training["pit_lane_time_s"].isna() & (training["result_status"].isin(["Finished", '+1 Lap', '+2 Laps', '+3 Laps']))
training.loc[mask, "pit_lane_time_s"] = 0

# For those who didn't finish, we'll leave it as is until now

##### Missing track status code

Column: track_status_code

In [155]:
training[training['track_status_code'].isna()] 
# 7 missing values
# I'm not filling it because 
## I can't guess what the status was 
### Gotta Google it to know
# BUT, those 7 cases didn’t finish the race… so we might do something
## Either we try to find out the track status by pulling out the same lap and see what other drivers have
### Ex:  the first case of missing track status is Leclerc, during the 1st lap of the Hungarian GP in 2018. What we’ll do is isolate that 1st lap of that race for everyone and look at the track status
### Apply this to all 7 missing values
## Leave them as is

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
5826,2018_12_LEC_001,2018_12,2018,12,Hungarian Grand Prix,Hungaroring,hungaroring,2018-07-29,LEC,16,...,False,0.8,103.0,82.539,False,False,False,False,0.8,983.7
8147,2018_18_ALO_005,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ALO,14,...,False,1.9,89.0,100.133,False,False,False,False,1.0,1005.8
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,3.3,121.0,99.500,False,False,False,False,1.4,1005.8
8748,2018_18_RIC_049,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,False,2.2,107.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
12141,2019_05_NOR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,NOR,4,...,False,1.8,48.0,79.681,False,False,False,False,1.3,1006.5
12515,2019_05_STR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,STR,18,...,False,1.8,48.0,79.681,False,False,False,False,1.3,1006.5
21037,2019_19_MAG_053,2019_19,2019,19,United States Grand Prix,Circuit of the Americas,austin,2019-11-03,MAG,20,...,False,1.3,189.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [156]:
# 1st missing value
# Leclerc, Hungarian GP 2018, 1st lap

training[(training['race_id'] == '2018_12') & (training['lap_number'] == 1)] 
# we can see that the track status for this lap is 1 all along (except for Leclerc's missing value)
# so here we'll just fill it with 1

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_12")
training.loc[mask, "track_status_code"] = '1'

In [157]:
# 2nd missing value
# Alonso, US GP 2018, 5th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 5)] 
# Same situation as Leclerc
# We'll proceed the same way

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_18") & (training['lap_number'] == 5)
training.loc[mask, "track_status_code"] = '1'

In [158]:
# 3rd missing value
# GrosJean, US GP 2018, 12th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 12)] 
# It's a bit more complicated here

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
8159,2018_18_BOT_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,BOT,77,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8215,2018_18_ERI_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ERI,9,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8270,2018_18_GAS_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GAS,10,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,3.3,121.0,99.5,False,False,False,False,1.4,1005.8
8328,2018_18_HAM_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HAM,44,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8384,2018_18_HAR_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HAR,28,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8439,2018_18_HUL_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HUL,27,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8495,2018_18_LEC_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,LEC,16,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8526,2018_18_MAG_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,MAG,20,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8
8582,2018_18_OCO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,OCO,31,...,False,0.4,163.0,99.5,False,False,False,False,1.4,1005.8


In [159]:
# 4th missing value
# Ricciardo, US GP 2018, 49th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 49)] 

# Same situation as Leclerc and Alonso
# We'll proceed the same way

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_18") & (training['lap_number'] == 49)
training.loc[mask, "track_status_code"] = '1'

In [160]:
# 5th & 6th missing values
# Norris & Stroll, Spanish GP 2019, 45th lap

training[(training['race_id'] == '2019_05') & (training['lap_number'] == 45)] 

# Also a bit complicated

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
11416,2019_05_ALB_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,ALB,23,...,False,2.5,288.0,79.681,False,False,False,False,1.3,1006.5
11482,2019_05_BOT_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,BOT,77,...,False,2.5,275.0,79.681,False,False,False,False,1.3,1006.5
11548,2019_05_GAS_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GAS,10,...,False,2.5,288.0,79.681,False,False,False,False,1.3,1006.5
11614,2019_05_GIO_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GIO,99,...,False,1.4,48.0,79.681,False,False,False,False,1.3,1006.5
11680,2019_05_GRO_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GRO,8,...,False,2.5,288.0,79.681,False,False,False,False,1.3,1006.5
11746,2019_05_HAM_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,HAM,44,...,False,2.5,275.0,79.681,False,False,False,False,1.3,1006.5
11812,2019_05_HUL_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,HUL,27,...,False,1.8,48.0,79.681,False,False,False,False,1.3,1006.5
11878,2019_05_KUB_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,KUB,88,...,False,2.0,19.0,79.681,False,False,False,False,1.3,1006.5
11943,2019_05_KVY_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,KVY,26,...,False,2.5,288.0,79.681,False,False,False,False,1.3,1006.5
12009,2019_05_LEC_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,LEC,16,...,False,2.5,288.0,79.681,False,False,False,False,1.3,1006.5


In [161]:
# 7th missing value
# Magnussen, US GP 2019, 53th lap

training[(training['race_id'] == '2019_19') & (training['lap_number'] == 53)]

# Same situation as Leclerc, Alonso, and Ricciardo
# We'll proceed the same way

mask = training["track_status_code"].isna() & (training["race_id"] == "2019_19") & (training['lap_number'] == 53)
training.loc[mask, "track_status_code"] = '1'

In [162]:
training[training['track_status_code'].isna()]
# Those are the 3 cases where it's a bit tricky
# And because of that, we can't tell what is True and what is False for the following columns:
# green, yellow, safety_car, virtual_sc & red_flag

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,3.3,121.0,99.500,False,False,False,False,1.4,1005.8
12141,2019_05_NOR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,NOR,4,...,False,1.8,48.0,79.681,False,False,False,False,1.3,1006.5
12515,2019_05_STR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,STR,18,...,False,1.8,48.0,79.681,False,False,False,False,1.3,1006.5


##### Missing green (flag I think)

Column: green

In [163]:
training[training['green']] # boolean column
training[training['green'].isna()] # 7 missing values

# we can use the track status
mask = training["green"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "green"] = True

##### Missing yellow (flag)

Column: yellow

In [164]:
training[training['yellow'].isna()]

mask = training["yellow"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "yellow"] = False

##### Missing safety car

Column: safety_car

In [165]:
training[training['safety_car'].isna()]
mask = training["safety_car"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "safety_car"] = False

##### Missing virtual safety car

Column: virtual_sc

In [166]:
training[training['virtual_sc'].isna()]
mask = training["virtual_sc"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "virtual_sc"] = False

##### Missing red flag

Column: red_flag

In [167]:
training[training['red_flag'].isna()]
mask = training["red_flag"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "red_flag"] = False


##### Missing pending penalties

COLUMN : penalty_pending_s

DESCRIPTION : Time penalty pending: seconds announced, not yet served. Can be served at a pit stop; otherwise added to the race time at the end.

In [ ]:
training[training['penalty_pending_s'].isna()] # 940 missing values
# All 

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,6.6,312.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,3.8,292.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.2,290.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.1,308.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [186]:
missing_penalty = training.loc[training["penalty_pending_s"].isna()]

missing_penalty.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


So all the missing values are in a single race: 2018 Australian GP.

Easiest thing is to say that no one received a penalty.

But this is something we cannot really assume. We don't really know what happened

So for now we'll just leave the missing values

##### Missing pending drive through

COLUMN : drive_through_pending

DESCRIPTION : Drive-through penalty pending (must be served in the pit lane within 3 laps)

In [ ]:
training[training['drive_through_pending'].isna()] # 940 missing values


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,6.6,312.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,3.8,292.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.2,290.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.1,308.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [188]:
missing_drive_through = training.loc[training["drive_through_pending"].isna()]

missing_drive_through.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same conclusion as for penalty_pending_s

All the missing values are in a single race: 2018 Australian GP.

Easiest thing is to say that no one received a drive through penalty.

But this is something we cannot really assume. We don't really know what happened

So for now we'll just leave the missing values

##### Missing pending stop go

COLUMN : stop_go_pending

DESCRIPTION : Stop-and-go penalty pending (must be served in the pit lane within 3 laps).

In [ ]:
training[training['stop_go_pending'].isna()] # 940 missing values


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,6.6,312.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,3.8,292.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.2,290.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.1,308.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [190]:
missing_stop_go = training.loc[training["stop_go_pending"].isna()]

missing_stop_go.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

##### Missing black & orange flag

COLUMN : black_orange_flag

DESCRIPTION : Car shown the black-and-orange flag (damage: must pit) during this lap.

In [ ]:
training[training['black_orange_flag'].isna()] # 940 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,6.6,312.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,3.8,292.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.2,290.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,4.1,308.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [197]:
missing_black_orange_flag = training.loc[training["black_orange_flag"].isna()]

missing_black_orange_flag.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Also Australian GP 2018

A black-and-orange flag is shown during a lap to tell a driver that their car has a mechanical problem or damage and must pit immediately.

2 options:
- The problem could be fixed and the driver could continue racing 
- Retire the car if the fault cannot be safely fixed

We cannot just deduce what happened during the race in this case.

So for now we'll just leave the missing values

##### Missing time penalties announced so far

COLUMN : time_penalties_announced_so_far

DESCRIPTION : Number of time penalties announced for this car so far in the race.

In [203]:
training[training['time_penalties_announced_so_far'].isna()] # 940 missing values

missing_time_penalties_announced_so_far= training.loc[training["time_penalties_announced_so_far"].isna()]

missing_time_penalties_announced_so_far.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

##### Missing black & white flags

COLUMN : black_white_flags_so_far

DESCRIPTION : Black-and-white flags (formal warnings) shown to this car so far.

In [202]:
training[training['black_white_flags_so_far'].isna()] # 940 missing values

missing_black_white_flags = training.loc[training["black_white_flags_so_far"].isna()]

missing_black_white_flags.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing as the black-and-orange flag

##### Missing track limit deletions so far

COLUMN : track_limit_deletions_so_far

DESCRIPTION : Lap times of this car deleted for track limits so far.

In [205]:
training[training['track_limit_deletions_so_far'].isna()] # 940 missing values

missing_track_limit_deletions = training.loc[training["track_limit_deletions_so_far"].isna()]

missing_track_limit_deletions.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

---

##### Missing fastest lap L+14

COLUMN : fastest_lap_Lplus14_s

DESCRIPTION : Fastest lap time of any car on race lap L + 14 (L = this row's lap).

In [ ]:
training[training['fastest_lap_Lplus14_s'].isna()] # 22536 missing values



,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
44,2018_01_ALO_045,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,5.3,312.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
45,2018_01_ALO_046,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.9,300.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
46,2018_01_ALO_047,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.0,313.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
47,2018_01_ALO_048,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.6,318.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
48,2018_01_ALO_049,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,5.2,278.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104752,2026_15_VER_047,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,1.7,357.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104753,2026_15_VER_048,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,2.2,54.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104754,2026_15_VER_049,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,0.6,353.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN
104755,2026_15_VER_050,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,0.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN


##### Missing safety car L+14

COLUMN : safety_car_Lplus14

DESCRIPTION : Safety car out on race lap L + 14 (any car's lap).

##### Missing virtual safety car L+14

COLUMN : virtual_sc_Lplus14

DESCRIPTION : Virtual safety car on race lap L + 14.

##### Missing red flag L+14

COLUMN : red_flag_Lplus14

DESCRIPTION : Red flag on race lap L + 14.

##### Missing rainfall L+14

COLUMN : rainfall_Lplus14

DESCRIPTION : Rain detected by the weather station on race lap L + 14.

##### Missing wind speed L+14

COLUMN : wind_speed_Lplus14_ms

DESCRIPTION : Wind speed on race lap L + 14 (median over the cars on that lap).

##### Missing pressure L+14

COLUMN : pressure_Lplus14_mbar

DESCRIPTION : Air pressure on race lap L + 14 (median over the cars on that lap).

#### 3.2.1.2 Training Dataset (version 2)

In [170]:
'''# There's 3 missing values in the column "team". 
# By figuring out who's the driver and which year was it, we can fill out those values (no outside data needed, just some data manipulation)

training[training['team'].isna()]

## - Alonso (2018, Race 18, 5th lap)
## - Grosjean (2018, Race 18, 12th lap)
## - Ricciardo (2018, Race 18, 49th lap)


# The goal now is to figure out which team each one of those drivers belonged to during that race, 
# and use this to fill the missing value

training[(training['race_id'] == '2018_18') & (training['driver_code'].isin(['ALO','GRO', 'RIC']))]

## Alonso was at McLaren,
## GrosJean was at Haas F1 Team,
## and Ricciardo was at Red Bull Racing


# Fill the missing values with the actual teams

replacements = pd.Series(['McLaren', 'Haas F1 Team', 'Red Bull Racing'], index=[8147, 8316, 8748]) 
## Those are the row indexes to make sure there's no mix up when it comes to puting what team where
training['team'] = training['team'].fillna(replacements)

# Just to check that the values where indeed filled
training[(training['race_id'] == '2018_18') & (training['driver_code'].isin(['ALO','GRO', 'RIC']))] '''

'# There\'s 3 missing values in the column "team". \n# By figuring out who\'s the driver and which year was it, we can fill out those values (no outside data needed, just some data manipulation)\n\ntraining[training[\'team\'].isna()]\n\n## - Alonso (2018, Race 18, 5th lap)\n## - Grosjean (2018, Race 18, 12th lap)\n## - Ricciardo (2018, Race 18, 49th lap)\n\n\n# The goal now is to figure out which team each one of those drivers belonged to during that race, \n# and use this to fill the missing value\n\ntraining[(training[\'race_id\'] == \'2018_18\') & (training[\'driver_code\'].isin([\'ALO\',\'GRO\', \'RIC\']))]\n\n## Alonso was at McLaren,\n## GrosJean was at Haas F1 Team,\n## and Ricciardo was at Red Bull Racing\n\n\n# Fill the missing values with the actual teams\n\nreplacements = pd.Series([\'McLaren\', \'Haas F1 Team\', \'Red Bull Racing\'], index=[8147, 8316, 8748]) \n## Those are the row indexes to make sure there\'s no mix up when it comes to puting what team where\ntraining[\'t

Pour toutes les lignes où
1. le conducteur n'a pas fini la course (DONC result_status n'est ni Finished ni +x Lap ni Lapped)
2. Y a des valeurs manquantes 

Je remplis ces valeurs manquantes par des N/A

Et s'il a fini la course mais y a quand même des valeurs manquantes, j'essaye de les remplir

In [171]:
'''finished = (
    training["result_status"].isin(["Finished", "Lapped"])
    | training["result_status"].str.match(r"^\+\d+ Laps?$")
) 


not_finished = training[~finished]
finished = training[finished]'''

<>:3: SyntaxWarning: invalid escape sequence '\+'
<>:3: SyntaxWarning: invalid escape sequence '\+'
C:\Users\laila\AppData\Local\Temp\ipykernel_16676\3691401493.py:3: SyntaxWarning: invalid escape sequence '\+'
  | training["result_status"].str.match(r"^\+\d+ Laps?$")


'finished = (\n    training["result_status"].isin(["Finished", "Lapped"])\n    | training["result_status"].str.match(r"^\\+\\d+ Laps?$")\n) \n\n\nnot_finished = training[~finished]\nfinished = training[finished]'

In [172]:
'''finished[finished['result_status'] == '+15 Laps'] 
# Does +15 laps mean he finished the race? 
# Because he's not classified'''

"finished[finished['result_status'] == '+15 Laps'] \n# Does +15 laps mean he finished the race? \n# Because he's not classified"

In [173]:
'''# filter not_finished by keeping the last lap recorded for each driver in each race
# Logic behind it: since those drivers didn't finish the race for some reason
# I'm keeping only the last lap to deal with the missing values here as N/A as said

last_laps = (
    not_finished
    .sort_values("lap_number")
    .groupby(["race_id", "driver_code"])
    .tail(1)
    .sort_values(["race_id", "driver_code"])
)
last_laps.isna().sum()

# How it works
# - sort_values("lap_number") orders each driver’s laps from first to last.
# - groupby(["race_id", "driver_code"]) makes one group per driver per race. You need both keys, because the same driver appears in many races.
# .tail(1) keeps the last row of each group, which is the highest lap_number after the sort.'''

'# filter not_finished by keeping the last lap recorded for each driver in each race\n# Logic behind it: since those drivers didn\'t finish the race for some reason\n# I\'m keeping only the last lap to deal with the missing values here as N/A as said\n\nlast_laps = (\n    not_finished\n    .sort_values("lap_number")\n    .groupby(["race_id", "driver_code"])\n    .tail(1)\n    .sort_values(["race_id", "driver_code"])\n)\nlast_laps.isna().sum()\n\n# How it works\n# - sort_values("lap_number") orders each driver’s laps from first to last.\n# - groupby(["race_id", "driver_code"]) makes one group per driver per race. You need both keys, because the same driver appears in many races.\n# .tail(1) keeps the last row of each group, which is the highest lap_number after the sort.'

In [174]:
'''
# 1) fill the missing values in last_laps however you like
last_laps.loc[last_laps["position"].isna(), "position"] = 20

# 2) copy the column back to training, matching on the index
training.loc[last_laps.index, "position"] = last_laps["position"]
'''

'\n# 1) fill the missing values in last_laps however you like\nlast_laps.loc[last_laps["position"].isna(), "position"] = 20\n\n# 2) copy the column back to training, matching on the index\ntraining.loc[last_laps.index, "position"] = last_laps["position"]\n'

In [175]:
'''last_laps['next_compound'] = last_laps['next_compound'].fillna('NA') 
training.loc[last_laps.index, "next_compound"] = last_laps["next_compound"]'''
# turns out that if i run this, there's still missing values in the 'next_compound' column, 
# so I ran the next cell which fixed it

'last_laps[\'next_compound\'] = last_laps[\'next_compound\'].fillna(\'NA\') \ntraining.loc[last_laps.index, "next_compound"] = last_laps["next_compound"]'

In [176]:
'''not_finished['next_compound'] = not_finished['next_compound'].fillna('NA') 
training.loc[not_finished.index, "next_compound"] = not_finished["next_compound"]'''

'not_finished[\'next_compound\'] = not_finished[\'next_compound\'].fillna(\'NA\') \ntraining.loc[not_finished.index, "next_compound"] = not_finished["next_compound"]'

#### 3.2.2 Test Dataset

##### Replacing the missing teams

In [177]:
# Using isna()
test[test['team'].isna()]

# Here, 1 value is missing in the column "team" and it's the following drivers : 
# - Alonso (2018, Race 14, 18th lap)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
2875,2018_14_ALO_018,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.8,138.0,82.839,False,False,False,True,2.5,995.1


In [178]:
test[(test['race_id'] == '2018_14') & (test['driver_code'] == 'ALO')]

# Just to make sure he was also at McLaren

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
2866,2018_14_ALO_001,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,True,1.0,225.0,84.582,False,False,False,False,3.1,995.2
2867,2018_14_ALO_002,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,True,0.4,174.0,84.674,False,False,False,False,2.7,995.2
2868,2018_14_ALO_003,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.8,187.0,84.545,False,False,False,False,2.1,995.2
2869,2018_14_ALO_004,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.3,0.0,84.582,False,False,False,False,1.5,995.2
2870,2018_14_ALO_005,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.8,251.0,84.516,False,False,False,False,3.0,995.2
2871,2018_14_ALO_006,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.5,317.0,84.448,False,False,False,False,2.7,995.2
2872,2018_14_ALO_007,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.7,197.0,83.611,False,False,False,False,2.6,995.2
2873,2018_14_ALO_008,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,2.6,181.0,83.953,False,False,False,False,0.9,995.1
2874,2018_14_ALO_009,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,2.3,181.0,83.535,False,False,False,False,2.7,995.1
2875,2018_14_ALO_018,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.8,138.0,82.839,False,False,False,True,2.5,995.1


In [179]:
# Fill the missing value with the actual team
test['team'] = test['team'].fillna('McLaren')

# Checking if the missing value was successefully replaced by the actual team name
test[(test['race_id'] == '2018_14') & (test['driver_code'] == 'ALO')] 
test[test['team'].isna()]
## Yes it's good

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
